# TREAD-ONE NB07 — data and protocol gate

**CPU only. Collects richer CSV evidence; no neural inference or training.**
This implements the next data/identity/protocol milestone after NB06. It audits
the existing 570 images / 12 original tyres, registers supplied new intake,
and records whether a prospective dataset is ready. Empty intake correctly
produces a completed audit with `prospective_training_ready=false`.

## Kaggle setup

1. Accelerator **None / CPU**; Internet **ON**.
2. Attach `shanmuk4622/tire-dataset-prepared-phase2` directly from Kaggle.
3. Optionally attach one new intake dataset containing `INTAKE_VERSION.json`,
   the tyre/session/image CSVs, `incoming/` native files and optional actual
   `protocol_policy.json`. With no new dataset, the bundled empty templates
   are used. Confirmed clip-parent mappings can be added to the resolution CSV.
4. Keep **1 GiB free**. Source images stay read-only. About 120 pinned training
   CSV shards are fetched for saved-exposure aggregation; no model-weight ZIP
   or full previous 2 GB experiment download is needed. No 1 TB scratch assumed.
5. Enable Kaggle Secret **HF_TOKEN** for final publication to
   `Shanmuk4622/tyre-wear-study`, then **Run All**. No package installation.

The CSVs contain actual source registries, fixed-grid quality descriptors,
native mask/point geometry and provenance, similarity review candidates,
conservative identity components, class/subject support, saved training exposure,
validated intake and explicit readiness gaps. These are not new independent
tyres or additional physical wear ground truth. Quality features are audit
outputs and are not silently fed to the models.

## Resume and output policy

Each completed source-image descriptor is cached atomically against source file,
annotation, code and environment hashes. Verified completed exposure downloads
are reused. An incomplete image/file is recomputed/downloaded on rerun. Input
CSVs/native files are never edited; derived records live in a versioned output.
An audit result can be useful and publishable while prospective training is not
ready. Required data, labels/identity, annotation and frozen-policy conditions
remain visible; unknown inputs are not filled with fake measurements.

Tables are directly browsable CSV. JSON schema/contracts, PNG figures and a
self-contained HTML explorer supplement them. Publication is additive under
`phase3/tread-one/data-gate/v1/<contract>`, uses bounded immutable batches and
deduplicates verified files. The final cell attempts a consistent immediate
flush on a catchable Stop/error, respecting HF server delays. Forced process
death cannot flush; completed local cache/files remain reusable.

No prospective source upload or new model is triggered automatically. Register
actual data and freeze meaningful subject counts/selection criteria before a
later model fit. Similarity is review triage, not physical identity evidence.


In [1]:
import base64,gzip,json,os,signal,subprocess,sys,shutil
from pathlib import Path
from filelock import FileLock
WORK=Path('/kaggle/working/treadone_nb07_data_gate')
WORK.mkdir(parents=True,exist_ok=True)
assert shutil.disk_usage(WORK).free>1024**3,'Need at least 1 GiB free for the gate/cache; source images remain in Kaggle Input.'
sources=json.loads(gzip.decompress(base64.b64decode('...')))
with FileLock(str(WORK/'NB07_SESSION.lock'),timeout=1):
    for name,value in sources.items():
        p=(WORK/name).resolve();assert p.is_relative_to(WORK.resolve())
        # Preserve an operator's already edited intake on rerun.
        if name.startswith('data_intake/') and p.exists():continue
        p.parent.mkdir(parents=True,exist_ok=True);p.write_bytes(base64.b64decode(value))
(WORK/'data_intake/incoming').mkdir(exist_ok=True)
candidates=[]
for p in Path('/kaggle/input').rglob('INTAKE_VERSION.json'):
    if json.loads(p.read_text()).get('schema')=='tyre-prospective-intake-v1':candidates.append(p.parent)
assert len(candidates)<=1,'Attach at most one prospective intake dataset or select it explicitly in INTAKE.'
INTAKE=candidates[0] if candidates else WORK/'data_intake'
print('CPU data gate ready. Intake:',INTAKE,'; no model training or inference is launched.')
def launch(name,*args):
    with FileLock(str(WORK/'NB07_SESSION.lock'),timeout=1):
        child=subprocess.Popen([sys.executable,'-B','-u',str(WORK/'data_gate'/name),*map(str,args)],start_new_session=True)
        try:
            while True:
                try:code=child.wait(timeout=15);break
                except subprocess.TimeoutExpired:pass
        except KeyboardInterrupt:
            if child.poll() is None:
                os.killpg(child.pid,signal.SIGINT)
                for i in range(60):
                    try:child.wait(timeout=5);break
                    except subprocess.TimeoutExpired:
                        if i%4==0:print('Allow completed-file cleanup or the publication flush to finish.')
            raise
        assert code==0,'Stage stopped; preserve logs and rerun to reuse completed image/file caches.'


CPU data gate ready. Intake: /kaggle/working/treadone_nb07_data_gate/data_intake ; no model training or inference is launched.


## Collect image, geometry, identity and training-exposure CSV records

In [2]:
launch('run_gate.py','--intake',INTAKE)

Source image/mask/point descriptors verified 50 / 570
Source image/mask/point descriptors verified 100 / 570
Source image/mask/point descriptors verified 150 / 570
Source image/mask/point descriptors verified 200 / 570
Source image/mask/point descriptors verified 250 / 570
Source image/mask/point descriptors verified 300 / 570
Source image/mask/point descriptors verified 350 / 570
Source image/mask/point descriptors verified 400 / 570
Source image/mask/point descriptors verified 450 / 570
Source image/mask/point descriptors verified 500 / 570
Source image/mask/point descriptors verified 550 / 570
Source image/mask/point descriptors verified 570 / 570
{
  "status": "audit_completed",
  "source_integrity_passed": true,
  "historical_split_consistent": true,
  "source_images": 570,
  "confirmed_existing_tyres": 12,
  "existing_video_clips": 3,
  "unresolved_video_mappings": 3,
  "containment_disagreement_images": 45,
  "new_intake_images": 0,
  "new_intake_tyre_claims": 0,
  "prospective_

## Render the interactive source explorer and prepare direct files

In [3]:
launch('report_gate.py')
launch('publish_gate.py','--prepare-only')

Rendered data gate report, 570 source previews and two scientific figures
{
  "status": "prepared_not_published",
  "path": "/kaggle/working/treadone_nb07_data_gate/data_gate_outputs/publication/dbb3cf3634d1",
  "contract": "dbb3cf3634d1c2b19034b32e7182a4deaab1fdb5dc094a605f733514009d6f91",
  "files": 46,
  "destination": "phase3/tread-one/data-gate/v1/dbb3cf3634d1c2b19034b32e7182a4deaab1fdb5dc094a605f733514009d6f91",
  "prospective_training_ready": false,
  "CSV_tables": 22
}


## Publish the completed data audit directly to HF

In [4]:
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN']=UserSecretsClient().get_secret('HF_TOKEN')
try:
    launch('publish_gate.py')
finally:
    os.environ.pop('HF_TOKEN',None)

{
  "status": "published_and_verified",
  "revision": "fe3d125a74ab1039b1ac34b503b01a495711e0ab",
  "destination": "phase3/tread-one/data-gate/v1/dbb3cf3634d1c2b19034b32e7182a4deaab1fdb5dc094a605f733514009d6f91",
  "files": 46,
  "uploaded_objects": 0,
  "zip_uploads": 0,
  "owner_released": true
}


## Read results and preserve the executed notebook

In [5]:
from IPython.display import display,FileLink
import pandas as pd
base=WORK/'data_gate_outputs'
latest=json.loads((base/'LATEST.json').read_text());folder=Path(latest['path'])
print((folder/'SUMMARY.json').read_text())
display(pd.read_csv(folder/'csv/historical_class_support.csv'))
display(pd.read_csv(folder/'csv/readiness_gaps.csv'))
display(FileLink(str(folder/'TREAD_ONE_Data_Gate.html')))
print((base/'PUBLICATION.json').read_text())
print('A false readiness flag is an expected audit finding when new intake/protocol is missing. No model was trained.')

{"confirmed_existing_tyres":12,"containment_disagreement_images":45,"csv_rows":7614,"csv_tables":22,"existing_video_clips":3,"historical_split_consistent":true,"new_independent_tyres_collected_by_agent":0,"new_inference":false,"new_intake_images":0,"new_intake_tyre_claims":0,"new_training":false,"prospective_training_ready":false,"source_images":570,"source_integrity_passed":true,"source_training_observations":{"baseline":23520,"shared_projection":23520},"status":"audit_completed","unresolved_video_mappings":3}


,confirmed_original_tyres,historical_split,human_geometry_images,images,is_prospective_blind_cohort,proxy_class,unmapped_existing_video_frames,weak_geometry_images
0,1,train,12,67,False,low,0,0
1,3,train,39,249,False,mid,152,152
2,4,train,19,70,False,high,0,0
3,1,validation,13,40,False,low,0,0
4,0,validation,0,0,False,mid,0,0
5,1,validation,12,41,False,high,0,0
6,1,test,13,62,False,low,0,0
7,0,test,0,0,False,mid,0,0
8,1,test,12,41,False,high,0,0


,code,detail
0,subject_minima_not_declared,Declare positive per-class subject minima for ...
1,prospective_support_incomplete,train/low: subject minimum not declared
2,prospective_support_incomplete,train/mid: subject minimum not declared
3,prospective_support_incomplete,train/high: subject minimum not declared
4,prospective_support_incomplete,validation/low: subject minimum not declared
5,prospective_support_incomplete,validation/mid: subject minimum not declared
6,prospective_support_incomplete,validation/high: subject minimum not declared
7,prospective_support_incomplete,test/low: subject minimum not declared
8,prospective_support_incomplete,test/mid: subject minimum not declared
9,prospective_support_incomplete,test/high: subject minimum not declared


/kaggle/working/treadone_nb07_data_gate/data_gate_outputs/runs/3eb4068aed9d/TREAD_ONE_Data_Gate.html

{"destination":"phase3/tread-one/data-gate/v1/dbb3cf3634d1c2b19034b32e7182a4deaab1fdb5dc094a605f733514009d6f91","files":46,"owner_released":true,"revision":"fe3d125a74ab1039b1ac34b503b01a495711e0ab","status":"published_and_verified","uploaded_objects":0,"zip_uploads":0}
A false readiness flag is an expected audit finding when new intake/protocol is missing. No model was trained.
